# Confronto OCSE

Confronti Italia-OCSE: totale, categorie e serie storiche con codici.


## Scaricamento dati

Esegui questa cella prima degli import di `utils_bilancio`. La cella usa solo Python standard per trovare il repository e, se serve, lancia `scripts/run_sections.py` per creare o aggiornare il JSON della sezione.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

SECTION_ID = "confronto_ocse"
REFRESH = False
FORCE_DOWNLOAD = False
repo_root = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "scripts" / "utils_bilancio").is_dir():
        repo_root = candidate
        break

if repo_root is None:
    raise ModuleNotFoundError("Impossibile trovare il repository Bilancio_pubblico.")

section_file = repo_root / "data" / "export" / "bilancio-pubblico" / "sections" / f"{SECTION_ID}.json"
command = [sys.executable, str(repo_root / "scripts" / "run_sections.py"), "--sections", SECTION_ID]
if REFRESH or FORCE_DOWNLOAD:
    command.append("--refresh")

env = os.environ.copy()
if SECTION_ID == "regioni" and SIOPE_YEARS:
    env["BILANCIO_PUBBLICO_SIOPE_YEARS"] = SIOPE_YEARS

if FORCE_DOWNLOAD or REFRESH or not section_file.exists():
    print("Eseguo:", " ".join(command))
    subprocess.run(command, cwd=repo_root, check=True, env=env)
else:
    print(f"Uso cache: {section_file}")


## Import e caricamento

Dopo che i file sono presenti, questa cella importa gli helper del repo e carica `section`, `status`, `frame` e l'eventuale `source_payload`.


In [ ]:
from pathlib import Path
import sys

SECTION_ID = "confronto_ocse"

if "repo_root" not in globals() or repo_root is None:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "scripts" / "utils_bilancio").is_dir():
            repo_root = candidate
            break
    else:
        raise ModuleNotFoundError("Impossibile trovare il repository Bilancio_pubblico.")

scripts_dir = repo_root / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from utils_bilancio.notebook.utils import setup_notebook_section

notebook_state = setup_notebook_section(
    section_id=SECTION_ID,
    refresh=False,
    force_download=False,
    include_source_payload=SECTION_ID == "italia",
)

section = notebook_state["section"]
status = notebook_state["status"]
frame = notebook_state["frame"]
source_payload = notebook_state.get("source_payload")


## Parametri principali

Nella cella **Scaricamento dati** puoi modificare:
- `REFRESH = False/True`: aggiorna OCSE e rigenera la sezione.
- `FORCE_DOWNLOAD = False/True`: forza una nuova materializzazione anche se il JSON esiste gia'.

Nelle celle di analisi puoi regolare:
- `VIEW`: vista fra quelle stampate in `OCSE_VIEW_OPTIONS`.
- `COUNTRY`: paese esattamente come appare in `OCSE_COUNTRIES`.
- `REVENUE_CODE` e `SPENDING_CODE`: codici aggregati Italia/OCSE.
- `PEER_REVENUE_CODE` e `PEER_SPENDING_CODE`: codici per serie storiche peer.
- `TOP`: numero di paesi/categorie da mostrare.

Esegui **Elenco opzioni disponibili** prima delle celle grafiche: i codici OCSE possono includere lettere, numeri e valori come `"_T"`.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
plt.style.use("ggplot")



In [ ]:
def rows_to_df(rows):
    return frame(rows or [])


def plot_bars(df, title, value_fields, top=10, x_col="codice"):
    if df.empty:
        print("Nessun dato")
        return
    field = None
    for name in value_fields:
        if name in df.columns:
            field = name
            break
    if field is None:
        print("Campo valore non trovato")
        return
    data = df.dropna(subset=[field]).sort_values(field, ascending=False).head(top)
    if data.empty:
        print("Nessun dato")
        return
    data.plot(x=x_col, y=field, kind="bar", figsize=(12, 5), legend=False)
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_series(df, title):
    if df.empty:
        print("Nessun dato per la serie")
        return
    base = df.sort_values("anno")
    base.plot(x="anno", y=["italia", "ocse", "differenza"], figsize=(12, 5))
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_peer_histories(peer_dict, code, value_title="valore", top=10):
    rows = rows_to_df(peer_dict.get(code, [])) if isinstance(peer_dict, dict) else pd.DataFrame()
    if rows.empty:
        print(f"Codice non trovato: {code}")
        return
    pivot = rows.pivot_table(index="anno", columns="paese", values="valore", aggfunc="mean")
    if pivot.empty:
        print("Serie vuota")
        return
    latest = pivot.iloc[-1].sort_values(ascending=False).head(top).sort_values()
    latest.plot(kind="barh", figsize=(12, 6))
    plt.title(f"{code}: {value_title} - ultimo anno")
    plt.tight_layout()
    plt.show()


## Elenco opzioni disponibili

Esegui questa cella prima delle celle dove imposti parametri:

- controlla i nomi ammessi
- usa esattamente i valori indicati
- verifica i codici/regioni/anni disponibili nella tua versione dati


In [ ]:
# Opzioni confronto OCSE - parametri ammessi

def print_values(header, values):
    print(header)
    if not values:
        print("  (nessun valore)")
        return
    for item in values:
        print(f" - {item}")


def to_sorted_int(values):
    clean = []
    for value in values:
        try:
            clean.append(int(value))
        except (TypeError, ValueError):
            pass
    return sorted(set(clean))


def format_lines(values, per_line=10):
    values = [str(item) for item in values]
    if not values:
        return ["(vuoto)"]
    values = sorted(set(values))
    return [", ".join(values[i : i + per_line]) for i in range(0, len(values), per_line)]


data = section.get("data", {})
views = [item.get("id") for item in section.get("available_views", []) if isinstance(item, dict)]

peer_revenue_total = rows_to_df(data.get("peer_revenue_total", []))
peer_spending_total = rows_to_df(data.get("peer_spending_total", []))
peer_inheritance = rows_to_df(data.get("peer_inheritance", []))

revenue_category_series = rows_to_df(data.get("revenue_category_series", []))
spending_category_series = rows_to_df(data.get("spending_category_series", []))
peer_revenue_category_series = data.get("peer_revenue_category_series", {}) or {}
peer_spending_category_series = data.get("peer_spending_category_series", {}) or {}

peer_revenue_codes = sorted([str(code) for code in peer_revenue_category_series.keys()])
peer_spending_codes = sorted([str(code) for code in peer_spending_category_series.keys()])

revenue_codes = []
if not revenue_category_series.empty and "codice" in revenue_category_series.columns:
    revenue_codes = sorted(set(revenue_category_series["codice"].dropna().astype(str).tolist()))
spending_codes = []
if not spending_category_series.empty and "codice" in spending_category_series.columns:
    spending_codes = sorted(set(spending_category_series["codice"].dropna().astype(str).tolist()))

ocse_countries = []
for frame in [peer_revenue_total, peer_spending_total, peer_inheritance]:
    if not frame.empty and "paese" in frame.columns:
        ocse_countries.extend(frame["paese"].dropna().astype(str).tolist())
ocse_countries = sorted(set(ocse_countries))

display_years = {}
for key, value in section.get("latest_years", {}).items():
    if value is not None:
        display_years[key] = value

year_options = to_sorted_int(display_years.values())

print("=== Parametri disponibili (Confronto OCSE) ===")
print_values("Views disponibili:", views)
print_values("Paesi disponibili (serie totali OCSE):", ocse_countries)
print(f"Anni aggregati disponibili: {display_years}")
print(f"Anni in scope: {year_options}")

print()
print("Codici entrate disponibili:")
print_values("Totale: %s" % len(revenue_codes), revenue_codes)
print()
print("Codici spesa disponibili:")
print_values("Totale: %s" % len(spending_codes), spending_codes)
print()
print("Codici peer revenue:")
print_values("Totale: %s" % len(peer_revenue_codes), peer_revenue_codes)
print()
print("Codici peer spesa:")
print_values("Totale: %s" % len(peer_spending_codes), peer_spending_codes)

print()
print("Esempio parametri:")
print('VIEW = "overview" if "overview" in OCSE_VIEW_OPTIONS else OCSE_VIEW_OPTIONS[0]')
print('COUNTRY = "Italia"')
print('REVENUE_CODE = OCSE_REVENUE_CODES[0] if OCSE_REVENUE_CODES else None')
print('PEER_REVENUE_CODE = OCSE_PEER_REVENUE_CODES[0] if OCSE_PEER_REVENUE_CODES else None')
print('Nota: i codici sono case-sensitive e possono includere lettere come "_T".')

OCSE_VIEW_OPTIONS = views
OCSE_YEAR_OPTIONS = display_years
OCSE_REVENUE_CODES = revenue_codes
OCSE_SPENDING_CODES = spending_codes
OCSE_PEER_REVENUE_CODES = peer_revenue_codes
OCSE_PEER_SPENDING_CODES = peer_spending_codes
OCSE_COUNTRY_OPTIONS = ocse_countries
OCSE_CODE_FORMAT = format_lines


## Disponibilità dati

Controlla le viste materializzate e i codici disponibili.

In [ ]:
print("Views:", [item.get("id") for item in section.get("available_views", [])])
for key, value in section.get("data", {}).items():
    if isinstance(value, list):
        print(f"{key}: {len(value)} righe")
    elif isinstance(value, dict):
        print(f"{key}: dict con {len(value)} codici")

data = section.get("data", {})
revenue_categories = rows_to_df(data.get("revenue_categories", []))
spending_categories = rows_to_df(data.get("spending_category_series", []))
if not revenue_categories.empty:
    display(revenue_categories.head(6))
if not spending_categories.empty:
    display(spending_categories.head(6))


## Confronti totali

Entrate, spesa e successioni Italia vs OCSE.

In [ ]:
peer_revenue_total = rows_to_df(data.get("peer_revenue_total", []))
peer_spending_total = rows_to_df(data.get("peer_spending_total", []))
peer_inheritance = rows_to_df(data.get("peer_inheritance", []))

for title, df in [
    ("Totale entrate", peer_revenue_total),
    ("Totale spesa", peer_spending_total),
    ("Successioni", peer_inheritance),
]:
    if df.empty:
        print(f"{title}: nessun dato")
        continue
    plot_bars(df, title, ["valore"], top=12, x_col="paese")
    year = int(pd.to_numeric(df["anno"], errors="coerce").max())
    latest = df[df["anno"] == year].sort_values("valore", ascending=False)
    if not latest.empty:
        latest.plot(x="paese", y="valore", kind="bar", figsize=(12, 4), legend=False)
        plt.title(f"{title} - {year}")
        plt.tight_layout()
        plt.show()


## Serie storiche per codici OCSE

Scegli codice e confronta le serie per area geografica.

In [ ]:
revenue_series = rows_to_df(data.get("revenue_category_series", []))
spending_series = rows_to_df(data.get("spending_category_series", []))
peer_revenue_series = data.get("peer_revenue_category_series", {})
peer_spending_series = data.get("peer_spending_category_series", {})

if not revenue_series.empty:
    rev_code = "_T"
    selected = revenue_series[revenue_series["codice"] == rev_code].sort_values("anno")
    if not selected.empty:
        plot_series(selected, f"Serie entrate {rev_code}", )

if not spending_series.empty:
    sp_code = "GF01"
    selected = spending_series[spending_series["codice"] == sp_code].sort_values("anno")
    if not selected.empty:
        plot_series(selected, f"Serie spesa {sp_code}")

if peer_revenue_series:
    sample_code = sorted(peer_revenue_series.keys())[0]
    plot_peer_histories(peer_revenue_series, sample_code, value_title="Entrate")

if peer_spending_series:
    sample_code = sorted(peer_spending_series.keys())[0]
    plot_peer_histories(peer_spending_series, sample_code, value_title="Spesa")


## Esplorazione rapida per codice

Genera liste top per tutti i codici disponibili nel dizionario map.

In [ ]:
peer_revenue_codes = sorted((data.get("peer_revenue_category_series") or {}).keys())
peer_spending_codes = sorted((data.get("peer_spending_category_series") or {}).keys())

print("Codici peer revenue:", len(peer_revenue_codes))
print("Codici peer spending:", len(peer_spending_codes))

for code in peer_revenue_codes[:8]:
    plot_peer_histories(data.get("peer_revenue_category_series", {}), code, value_title="Entrate")

for code in peer_spending_codes[:8]:
    plot_peer_histories(data.get("peer_spending_category_series", {}), code, value_title="Spesa")
